# Client GI Review (flawed reports)

Runs `with_errors_reports/*_flawed.md` to validate injected-error detection against `client.json`.

1. Set **CLIENT** and options in the config cell below.
2. Run the review cell.

Outputs: cost table per report + findings table (`injected_error`, `missed_injected`, `severity`).

**KPIs**
- Recall: **11/11** introduced flaws caught (any severity).
- Precision: **blocking flags ≈ 0** on corrected PASS reports (`INCLUDE_CORRECTED=True`).

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from client_review import (
    blocking_precision_on_corrected,
    checkpoint_lookup,
    cost_summary_rows,
    findings_rows,
    load_client,
    run_client_review,
)

# --- config ---
CLIENT = "ribkoff"
INCLUDE_CORRECTED = False  # flawed-only eval; set True to also run corrected_reports
INCLUDE_FLAWED = True
STRIP_COMMENTS = True
CHECKPOINT_LIMIT = None  # None = all checkpoints
MAX_WORKERS = 16
STRICT_BLOCKING = False  # True = BLOCKING also flags partial_unmatch

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/williambrun/Documents/test_GI_reports/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other 

In [ ]:
from IPython.display import display

batch = run_client_review(
    CLIENT,
    root=PROJECT_ROOT,
    include_corrected=INCLUDE_CORRECTED,
    include_flawed=INCLUDE_FLAWED,
    checkpoint_limit=CHECKPOINT_LIMIT,
    max_workers=MAX_WORKERS,
    strip_comments=STRIP_COMMENTS,
    strict_blocking=STRICT_BLOCKING,
)

client = load_client(CLIENT, PROJECT_ROOT)
cost_df = pd.DataFrame(cost_summary_rows(batch.report_summaries))
findings_df = pd.DataFrame(
    findings_rows(client, batch.report_summaries, checkpoint_lookup(batch.checkpoints_path))
)

injected_hits = int((findings_df["injected_error"] & ~findings_df["missed_injected"]).sum())
missed = int(findings_df["missed_injected"].sum())
blocking_count = sum(s.get("blocking_flags_count", s["flags_count"]) for s in batch.report_summaries)
total_count = sum(s.get("total_flags_count", s["flags_count"]) for s in batch.report_summaries)
precision = blocking_precision_on_corrected(batch.report_summaries)

print(f"Model: {batch.model}")
unable_count = sum(s.get("unable_to_check_count", 0) for s in batch.report_summaries)
print(
    f"Reports: {len(batch.report_summaries)} | "
    f"Blocking flags (KPI): {blocking_count} | Total flags: {total_count} | "
    f"Unable to check: {unable_count}"
)
print(f"Total tokens: {batch.total_tokens:,} | Total cost: ${batch.total_cost_usd:.4f}")
print(f"Injected errors caught: {injected_hits} | Missed: {missed}")
if precision["corrected_reports"]:
    print(
        f"Corrected PASS blocking flags: {precision['blocking_flags']} "
        f"across {precision['corrected_reports']} report(s)"
    )
print(f"Saved to: {batch.output_dir}\n")

display(cost_df)
display(
    findings_df[
        [
            "report",
            "report_set",
            "checkpoint_id",
            "match",
            "severity",
            "blocking",
            "injected_error",
            "injected_change",
            "missed_injected",
            "reason",
            "report_snippet",
        ]
    ]
)